In [1]:
#Load Cleaned Dataset
import pandas as pd

df = pd.read_csv("data/newsroom_analysis_ready.csv.gz")

In [2]:
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/dangeloba/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/dangeloba/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/dangeloba/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
#Processing
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

In [4]:
df = df.sample(50000, random_state=42)

import nltk

nltk.download('punkt')
nltk.download('punkt_tab')   # missing this

[nltk_data] Downloading package punkt to /Users/dangeloba/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/dangeloba/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
texts = df['text'].apply(preprocess)

In [6]:
#install gensim
!pip install gensim

In [7]:
#Build Document-Term Matrix
from gensim.corpora import Dictionary

dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

In [8]:
#Run LDA
from gensim.models import LdaModel

lda = LdaModel(corpus, num_topics=10, id2word=dictionary, passes=5)

In [9]:
#Assign Topic to Each Document
df['topic'] = [max(lda[doc], key=lambda x: x[1])[0] for doc in corpus]

In [10]:
#Compare
df.groupby(['topic', 'publisher_clean']).size()

topic  publisher_clean
0      9news.com          174
       abcnews             29
       aol                 29
       bbc                 41
       bloomberg           24
                         ... 
9      time               308
       tmz                 11
       usatoday           232
       washingtonpost     372
       wsj                195
Length: 299, dtype: int64

In [11]:
df.groupby(['topic', 'publisher_clean']).size().unstack().fillna(0)

publisher_clean,9news.com,abcnews,aol,bbc,bloomberg,bostonglobe,cbc,cnbc,cnn,forbes,...,reuters,sfgate,telegraph,theguardian,thesun,time,tmz,usatoday,washingtonpost,wsj
topic,,,,,,,,,,,,,,,,,,,,,
0,174.0,29.0,29.0,41.0,24.0,8.0,81.0,48.0,390.0,50.0,...,46.0,9.0,94.0,508.0,52.0,279.0,3.0,89.0,525.0,267.0
1,80.0,125.0,175.0,180.0,22.0,317.0,128.0,62.0,408.0,246.0,...,36.0,171.0,473.0,1057.0,381.0,705.0,425.0,405.0,965.0,426.0
2,58.0,86.0,82.0,8.0,68.0,29.0,59.0,34.0,358.0,51.0,...,48.0,40.0,34.0,210.0,11.0,223.0,10.0,226.0,956.0,254.0
3,23.0,20.0,31.0,39.0,31.0,193.0,19.0,175.0,85.0,268.0,...,13.0,33.0,64.0,75.0,15.0,132.0,9.0,157.0,143.0,315.0
4,9.0,42.0,128.0,7.0,4.0,26.0,9.0,6.0,41.0,16.0,...,1.0,51.0,12.0,33.0,10.0,77.0,69.0,87.0,38.0,34.0
5,26.0,17.0,33.0,25.0,3.0,19.0,118.0,11.0,83.0,48.0,...,3.0,147.0,29.0,317.0,91.0,57.0,77.0,538.0,367.0,119.0
6,36.0,17.0,38.0,20.0,114.0,128.0,30.0,665.0,73.0,520.0,...,81.0,16.0,147.0,118.0,15.0,127.0,12.0,137.0,297.0,541.0
7,393.0,77.0,59.0,23.0,5.0,38.0,92.0,24.0,326.0,23.0,...,11.0,74.0,31.0,341.0,116.0,152.0,342.0,242.0,853.0,133.0
8,171.0,42.0,69.0,130.0,12.0,49.0,86.0,29.0,199.0,28.0,...,13.0,152.0,86.0,199.0,43.0,137.0,24.0,186.0,276.0,173.0


In [12]:
df.groupby(['topic', 'publisher_clean']).size().groupby(level=0).apply(lambda x: x / x.sum())

topic  topic  publisher_clean
0      0      9news.com          0.038016
              abcnews            0.006336
              aol                0.006336
              bbc                0.008958
              bloomberg          0.005244
                                   ...   
9      9      time               0.069307
              tmz                0.002475
              usatoday           0.052205
              washingtonpost     0.083708
              wsj                0.043879
Length: 299, dtype: float64

In [14]:
from gensim.models import LdaModel

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=9,   # topic number
    passes=10
)

In [15]:
lda_model.print_topics()

[(0,
  '0.015*"company" + 0.013*"year" + 0.011*"said" + 0.011*"million" + 0.010*"ha" + 0.010*"market" + 0.008*"percent" + 0.007*"price" + 0.007*"billion" + 0.007*"business"'),
 (1,
  '0.018*"wa" + 0.008*"one" + 0.007*"like" + 0.006*"ha" + 0.005*"time" + 0.005*"say" + 0.005*"year" + 0.004*"new" + 0.004*"show" + 0.003*"life"'),
 (2,
  '0.015*"wa" + 0.015*"game" + 0.011*"team" + 0.009*"said" + 0.008*"season" + 0.008*"ha" + 0.008*"player" + 0.006*"first" + 0.006*"year" + 0.006*"one"'),
 (3,
  '0.013*"said" + 0.011*"ha" + 0.009*"wa" + 0.008*"government" + 0.008*"country" + 0.007*"state" + 0.006*"united" + 0.006*"military" + 0.006*"war" + 0.005*"would"'),
 (4,
  '0.008*"people" + 0.008*"say" + 0.007*"ha" + 0.007*"one" + 0.006*"would" + 0.005*"year" + 0.005*"make" + 0.005*"said" + 0.005*"like" + 0.005*"get"'),
 (5,
  '0.028*"wa" + 0.023*"said" + 0.008*"court" + 0.008*"post" + 0.007*"case" + 0.007*"ha" + 0.007*"law" + 0.006*"report" + 0.006*"told" + 0.005*"comment"'),
 (6,
  '0.013*"obama" + 0